# Retomar a etapa 6 sem baixar os dados novamente

Abra este arquivo para copiar a única célula de código abaixo. Cole-a no notebook 14 do Kaggle que apresentou o erro, na mesma sessão em que as etapas 1–5 já rodaram, e execute. Ela aplica a correção e continua automaticamente até gerar o pacote final. Não execute este arquivo em uma sessão vazia.


In [ ]:
# Executar esta célula na MESMA sessão do notebook 14 que parou na etapa 6.
# Ela aplica a correção e executa automaticamente as etapas 6, 7 e 8.
from pathlib import Path
from datetime import datetime,timezone
import hashlib

obrigatorios=['MODULOS_EMBUTIDOS','avaliacao14','relatorios14','CONFIG_14','COORTES',
    'PASTA_RESULTADOS','PASTA_CODIGO','PESO_B','EXPECTED','AMBIENTE','DEVICE','FROZEN_FILES',
    'BUNDLE','politica','avaliacao_base','enviar_resultados','registrar_falha','core','dados12','YOLO','torch']
ausentes=[nome for nome in obrigatorios if nome not in globals()]
if ausentes:
    raise RuntimeError('Esta célula exige a sessão aberta do notebook 14 com as etapas 1–5 executadas. '
        'Copie-a para essa sessão; se ela foi encerrada, use o notebook 14 completo corrigido. Ausentes: '+', '.join(ausentes))

CODIGO_CORRIGIDO = {'conf14_evaluation.py': '"""Varredura de confiança: uma inferência por imagem, política V2 fixa."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport diag11_core as core\nimport conf14_data as data\nfrom exp12_protocol import judge\n\nTHRESHOLDS=(.05,.10,.15,.20,.25,.30,.40)\nPREDICT_ARGS={\'conf\':.001,\'imgsz\':640,\'rect\':True,\'iou\':.70,\'max_det\':300,\'half\':False,\'batch\':1}\nVERSION=\'14-confianca-automatica-1.0\'\n\ndef point_name(conf):return \'conf_\'+f\'{conf:.2f}\'.replace(\'.\',\'\')\n\n\ndef cache_request(records,weight_sha,environment):\n    if not records or len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Cache exige IDs únicos.\')\n    if any(r[\'role\'] not in {\'home_val\',\'val_nova\'} for r in records):raise ValueError(\'Cache somente para validação.\')\n    signature=[{k:r[k] for k in [\'id\',\'sha256\',\'label_sha256\',\'width\',\'height\',\'known_mask\',\'role\']} for r in records]\n    return {\'versao\':VERSION,\'peso_sha256\':weight_sha,\'parametros\':PREDICT_ARGS,\n            \'ambiente\':environment,\'coorte\':records[0][\'coorte\'],\'imagens\':len(records),\n            \'entradas_sha256\':hashlib.sha256(core.canonical(signature).encode()).hexdigest()}\n\ndef read_cache(path,records):\n    expected={r[\'id\']:r for r in records};result={}\n    with gzip.open(path,\'rt\',encoding=\'utf-8\') as f:\n        for line in f:\n            row=json.loads(line);iid=row[\'id\']\n            if iid not in expected or iid in result:raise ValueError(\'Cache com ID desconhecido ou repetido.\')\n            r=expected[iid]\n            if row[\'image_sha256\']!=r[\'sha256\'] or row[\'label_sha256\']!=r[\'label_sha256\']:\n                raise ValueError(\'Cache pertence a outros bytes de imagem/label.\')\n            validate_predictions(row[\'predicoes\'],r);result[iid]=row[\'predicoes\']\n    if result.keys()!=expected.keys():raise ValueError(\'Cache incompleto.\')\n    return result\n\n\ndef event(record,kind,g,p,overlap,pred_low,conf):\n    row=core.record_event(record,\'B_e03\',kind,g,p,overlap,pred_low)\n    row[\'limiar_confianca\']=conf\n    row[\'melhor_confianca_mesma_classe\']=None\n    if kind==\'objeto_nao_detectado\':\n        flags=[v for v in row[\'indicios\'].split(\'|\') if v and v not in {\'deteccao_abaixo_da_confianca_025\',\'sobreposicao_com_previsao_de_outra_classe\',\'omissao_sem_indicio_automatico\'}]\n        matrix=core.iou_matrix([g],pred_low)[0]\n        same=[i for i,p0 in enumerate(pred_low) if p0[\'classe_id\']==g[\'classe_id\']]\n        if same:\n            best=max(same,key=lambda i:matrix[i]);row[\'melhor_confianca_mesma_classe\']=pred_low[best][\'confianca\']\n            if matrix[best]>=.5 and pred_low[best][\'confianca\']<conf:flags.append(\'deteccao_abaixo_do_limiar_avaliado\')\n        if any(matrix[i]>=.5 and p0[\'classe_id\']!=g[\'classe_id\'] and p0[\'confianca\']>=conf for i,p0 in enumerate(pred_low)):\n            flags.append(\'sobreposicao_com_previsao_de_outra_classe\')\n        row[\'indicios\']=\'|\'.join(flags or [\'omissao_sem_indicio_automatico\'])\n    return row\n\ndef analyse(record,pred_low,conf,policy,matcher):\n    if record[\'role\'] not in {\'home_val\',\'val_nova\'} or conf not in THRESHOLDS:\n        raise ValueError(\'Papel ou limiar fora do protocolo.\')\n    known=int(record[\'known_mask\'])\n    gt=core.with_policy([g for g in record[\'gt\'] if known&(1<<g[\'classe_id\'])],record[\'width\'],record[\'height\'],policy)\n    low=[p for p in pred_low if known&(1<<p[\'classe_id\'])]\n    all_pred=core.with_policy([p for p in pred_low if p[\'confianca\']>=conf],record[\'width\'],record[\'height\'],policy)\n    pred=[p for p in all_pred if known&(1<<p[\'classe_id\'])]\n    pairs,missing,extras=matcher(gt,pred)\n    events=[event(record,\'correspondencia\',gt[gi],pred[pi],float(iou),low,conf) for gi,pi,iou in pairs]\n    events += [event(record,\'objeto_nao_detectado\',gt[gi],None,None,low,conf) for gi in missing]\n    events += [event(record,\'previsao_nao_associada\',None,pred[pi],None,low,conf) for pi in extras]\n    messages=[policy.gerar_mensagem(p,\'medio\') for p in all_pred if core.alert_on(p)]\n    if any(not isinstance(s,str) or not s.strip() for s in messages):raise RuntimeError(\'Alerta sem mensagem.\')\n    im={\'coorte\':record[\'coorte\'],\'imagem\':record[\'name\'],\'id\':record[\'id\'],\'modelo\':\'B_e03\',\'limiar_confianca\':conf,\n        \'tp_alerta\':sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'fn_alerta\':sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in events),\n        \'fp_alerta\':sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'alta_ref\':sum(g[\'prioridade\']==\'alto\' for g in gt),\n        \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in events),\n        \'tp_detector\':len(pairs),\'fn_detector\':len(missing),\'fp_detector\':len(extras),\n        \'gt_avaliaveis\':len(gt),\'gt_ignorados\':len(record[\'gt\'])-len(gt),\n        \'predicoes_ignoradas_conf\':len(all_pred)-len(pred),\n        \'cobertura_completa_alertas\':known&core.ALERT_MASK==core.ALERT_MASK,\n        \'ha_alerta_ref\':any(core.alert_on(g) for g in gt),\'ha_alerta_pred\':any(core.alert_on(p) for p in pred),\n        \'mensagem_alerta\':\'\\n\'.join(messages)}\n    return im,events\n\ndef summarise(images,events,conf):\n    if not images:raise ValueError(\'Validação vazia.\')\n    total=lambda k:sum(r[k] for r in images)\n    complete=[r for r in images if r[\'cobertura_completa_alertas\']]\n    summary={\'modelo\':\'B_e03\',\'coorte\':images[0][\'coorte\'],\'imagens\':len(images),\'conf_alertas\':conf,\n        \'alertas_por_objeto\':core.alert_metrics(total(\'tp_alerta\'),total(\'fn_alerta\'),total(\'fp_alerta\')),\n        \'alta_ref\':total(\'alta_ref\'),\'alta_sem_alerta\':total(\'alta_sem_alerta\'),\n        \'taxa_alta_sem_alerta\':core.ratio(total(\'alta_sem_alerta\'),total(\'alta_ref\')),\n        \'detector_no_limiar\':core.alert_metrics(total(\'tp_detector\'),total(\'fn_detector\'),total(\'fp_detector\')),\n        \'gt_ignorados_sem_cobertura\':total(\'gt_ignorados\'),\'predicoes_ignoradas_conf\':total(\'predicoes_ignoradas_conf\'),\n        \'imagens_com_cobertura_completa_alertas\':len(complete),\n        \'alertas_por_imagem_cobertura_completa\':core.alert_metrics(\n            sum(r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete),\n            sum(r[\'ha_alerta_ref\'] and not r[\'ha_alerta_pred\'] for r in complete),\n            sum(not r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete)),\n        \'referencia\':\'Concordância com política V2 nas caixas; não valida risco físico.\',\n        \'nao_e_openimages_challenge\':True,\'ap_recalculada\':False}\n    classes=[]\n    for cid,name in enumerate(core.NAMES):\n        part=[e for e in events if e[\'classe_id\']==cid];n=sum(e[\'bbox_ref\']!=\'\' for e in part)\n        classes.append({\'classe_id\':cid,\'classe\':name,\'gt_avaliaveis\':n,\'suporte_reduzido\':n<30,\'conf_alertas\':conf,\n            **core.alert_metrics(sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part),\n                                 sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in part),\n                                 sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part)),\n            \'alta_ref\':sum(e[\'prioridade_ref\']==\'alto\' for e in part),\n            \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in part)})\n    return summary,classes\n\ndef evaluate_cache(records,cache,conf,policy,matcher,output=None):\n    if cache.keys()!={r[\'id\'] for r in records}:raise ValueError(\'Cache não corresponde aos IDs da validação.\')\n    images=[];events=[]\n    for r in records:\n        im,ev=analyse(r,cache[r[\'id\']],conf,policy,matcher);images.append(im);events.extend(ev)\n    summary,classes=summarise(images,events,conf)\n    if output is not None:\n        output=Path(output);output.mkdir(parents=True,exist_ok=True)\n        core.save_json(output/\'metricas.json\',summary)\n        core.save_csv(output/\'metricas_por_classe.csv\',classes)\n        core.save_csv(output/\'alertas_por_imagem.csv\',images)\n        core.save_csv(output/\'avaliacao_objetos.csv\',events)\n    return {\'summary\':summary,\'classes\':classes,\'images\':images,\'events\':events}\n\ndef parse_current_events(events):\n    result=[]\n    for r in events:\n        v=dict(r)\n        for key in [\'bbox_ref\',\'bbox_pred\']:v[key]=json.loads(v[key]) if v[key] else None\n        result.append(v)\n    return result\n\ndef check_reproduction(result,original,cohort):\n    old={r[\'id\']:r for r in original[\'images\']};new={r[\'id\']:r for r in result[\'images\']}\n    if old.keys()!=new.keys():raise ValueError(\'Reprodução exige os mesmos IDs.\')\n    rows=[]\n    fields=[\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']\n    for iid in sorted(old):\n        for key in fields:\n            key_new=\'predicoes_ignoradas_conf\' if key==\'predicoes_ignoradas_conf_025\' else key\n            a,b=old[iid][key],new[iid][key_new]\n            rows.append({\'coorte\':cohort,\'id\':iid,\'metrica\':key,\'notebook12\':a,\'notebook14\':b,\'igual\':a==b})\n    old_gt=data.gt_map(original[\'events\']);new_gt=data.gt_map(parse_current_events(result[\'events\']))\n    if old_gt.keys()!=new_gt.keys():raise ValueError(\'Referências GT mudaram na reprodução.\')\n    object_rows=[]\n    for key,a in old_gt.items():\n        b=new_gt[key]\n        fields_gt=[\'evento\',\'prioridade_ref\',\'prioridade_pred\',\'ha_alerta_ref\',\'ha_alerta_pred\']\n        diffs=[field for field in fields_gt if a[field]!=b[field]]\n        object_rows.append({\'coorte\':cohort,\'id\':key[0],\'classe\':a[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                            \'campos_diferentes\':\'|\'.join(diffs),\'igual\':not diffs})\n    return {\'coorte\':cohort,\'checagens_imagem\':len(rows),\'divergencias_imagem\':sum(not r[\'igual\'] for r in rows),\n            \'objetos_GT\':len(object_rows),\'divergencias_objetos\':sum(not r[\'igual\'] for r in object_rows),\n            \'reproduziu\':all(r[\'igual\'] for r in rows+object_rows)},rows,object_rows\n\ndef select_threshold(results,rule,reproduction_ok):\n    assessed=[]\n    for conf in THRESHOLDS:\n        home=results[\'home_val_historica\'][conf][\'summary\'];ext=results[\'openimages_val_nova\'][conf][\'summary\']\n        supported=all(s[\'alertas_por_objeto\'][k] is not None and math.isfinite(s[\'alertas_por_objeto\'][k])\n                      for s in [home,ext] for k in [\'precision\',\'recall\',\'f1\'])\n        if supported:\n            checks={**judge(home,ext,rule),\'metricas_com_suporte\':True}\n        else:\n            # Precisão sem nenhum alerta é indefinida; não vira zero ou aprovação.\n            checks={k:None for k in [\'home_precision_superior\',\'home_recall_preservado\',\'home_altas_preservadas\',\n                                    \'externa_precision_preservada\',\'externa_recall_preservado\',\'externa_altas_preservadas\']}\n            checks.update(elegivel=False,retencao_externa=False,metricas_com_suporte=False)\n        assessed.append({\'conf\':conf,\'home\':home,\'external\':ext,\'checks\':checks,\n                         \'comparavel_historico\':reproduction_ok,\'elegivel\':checks[\'elegivel\'] and reproduction_ok})\n    def ranking(r):\n        h=r[\'home\'][\'alertas_por_objeto\'];e=r[\'external\'][\'alertas_por_objeto\']\n        return (-h[\'precision\'],-h[\'recall\'],r[\'home\'][\'alta_sem_alerta\'],-e[\'f1\'],abs(r[\'conf\']-.25),-r[\'conf\'])\n    good=[r for r in assessed if r[\'elegivel\']]\n    chosen=min(good,key=ranking) if good else None\n    status=\'limiar_candidato_em_validacao\' if chosen else \'nenhum_limiar_elegivel\'\n    if not reproduction_ok:status=\'reproducao_divergente_sem_sugestao_de_limiar\'\n    return {\'status\':status,\'limiar_sugerido\':chosen[\'conf\'] if chosen else None,\n            \'comparabilidade_confirmada\':bool(reproduction_ok),\'assessed\':assessed,\n            \'limiar_galeria\':chosen[\'conf\'] if chosen else .10,\n            \'criterio_galeria_sem_elegivel\':\'0,10 fixado antes da inferência; somente diagnóstico.\',\n            \'modelo_em_uso_continua_baseline\':True,\'substituicao_automatica\':False}\n\ndef effects(results,bundle,cohort,cache):\n    old=data.gt_map(bundle[\'evaluations\'][\'candidato_10\'][cohort][\'events\'])\n    anchor=data.gt_map(parse_current_events(results[cohort][.25][\'events\']))\n    lost={k for k in anchor if old[k][\'ha_alerta_ref\'] and old[k][\'ha_alerta_pred\'] and not anchor[k][\'ha_alerta_pred\']}\n    changed=[];losses=[];priority=[]\n    for key in sorted(lost):\n        a,b=old[key],anchor[key]\n        if b[\'evento\']==\'correspondencia\':\n            priority.append({\'coorte\':cohort,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_candidato10\':a[\'prioridade_pred\'],\'prioridade_B_025\':b[\'prioridade_pred\'],\n                \'confianca_candidato10\':a[\'confianca\'],\'confianca_B_025\':b[\'confianca\'],\n                \'observacao\':\'Objeto ainda associado; alerta pode mudar pela geometria/prioridade.\'})\n    for conf in THRESHOLDS:\n        current=data.gt_map(parse_current_events(results[cohort][conf][\'events\']))\n        for key,b in current.items():\n            a=anchor[key]\n            if a[\'ha_alerta_ref\'] and a[\'ha_alerta_pred\']!=b[\'ha_alerta_pred\']:\n                changed.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                    \'alerta_B_025\':a[\'ha_alerta_pred\'],\'alerta_no_limiar\':b[\'ha_alerta_pred\'],\n                    \'prioridade_ref\':b[\'prioridade_ref\'],\'confianca_associada\':b[\'confianca\']})\n            if key not in lost:continue\n            same=[p for p in cache[key[0]] if p[\'classe_id\']==key[1]]\n            ious=core.iou_matrix([{\'bbox\':list(key[2])}],same)[0]\n            best=max(range(len(same)),key=lambda i:ious[i]) if same else None\n            losses.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_ref\':b[\'prioridade_ref\'],\'recuperado\':b[\'ha_alerta_pred\'],\n                \'evento_B_025\':a[\'evento\'],\'evento_no_limiar\':b[\'evento\'],\n                \'melhor_iou_cache_mesma_classe\':float(ious[best]) if best is not None else None,\n                \'confianca_dessa_proposta\':same[best][\'confianca\'] if best is not None else None,\n                \'limite\':\'Melhor IoU não garante associação nem alerta; coluna recuperado usa avaliação completa.\'})\n    return changed,losses,priority\n\n\ndef validate_predictions(pred,record):\n    """Valida estrutura; preserva coordenadas numéricas como no avaliador 11/12.\n\n    Área zero e limites da imagem são diagnósticos, não filtros do cache.\n    Nenhuma proposta é removida, recortada ou corrigida aqui.\n    """\n    totals={\'predicoes\':len(pred),\'area_zero\':0,\'fora_da_imagem\':0,\n            \'area_zero_na_grade\':0,\'fora_da_imagem_na_grade\':0,\'amostras\':[]}\n    for index,p in enumerate(pred):\n        reason=None\n        if not isinstance(p,dict):\n            raise ValueError(f"Previsão não é um dicionário: {record[\'id\']}; índice={index}; valor={p!r}")\n        cid=p.get(\'classe_id\');box=p.get(\'bbox\');score=p.get(\'confianca\')\n        if type(cid)!=int or not 0<=cid<len(core.NAMES):reason=\'classe_id inválida\'\n        elif p.get(\'classe\')!=core.NAMES[cid]:reason=\'nome da classe incompatível\'\n        elif type(score) not in (int,float) or not math.isfinite(score) or not .001-1e-8<=score<=1:\n            reason=\'confiança não finita ou fora da faixa de captura\'\n        elif not isinstance(box,(list,tuple)) or len(box)!=4:reason=\'bbox deve ter quatro coordenadas\'\n        elif not all(type(v) in (int,float) and math.isfinite(v) for v in box):reason=\'coordenada não numérica ou não finita\'\n        elif box[2]<box[0] or box[3]<box[1]:reason=\'coordenadas xyxy invertidas\'\n        if reason:\n            raise ValueError(f"Previsão incompatível com o cache: {record[\'id\']}; motivo={reason}; "\n                f"índice={index}; classe_id={cid!r}; confiança={score!r}; bbox={box!r}; "\n                f"dimensões=(w={record[\'width\']}, h={record[\'height\']})")\n        zero=box[2]==box[0] or box[3]==box[1]\n        outside=(min(box)<-.01 or box[2]>record[\'width\']+.01 or box[3]>record[\'height\']+.01)\n        totals[\'area_zero\']+=int(zero);totals[\'fora_da_imagem\']+=int(outside)\n        totals[\'area_zero_na_grade\']+=int(zero and score>=min(THRESHOLDS))\n        totals[\'fora_da_imagem_na_grade\']+=int(outside and score>=min(THRESHOLDS))\n        if (zero or outside) and len(totals[\'amostras\'])<5:\n            totals[\'amostras\'].append({\'indice\':index,\'classe\':p[\'classe\'],\'confianca\':score,\n                \'bbox\':list(box),\'area_zero\':zero,\'fora_da_imagem\':outside,\n                \'width\':record[\'width\'],\'height\':record[\'height\']})\n    return totals\n\ndef save_geometry_diagnostics(directory,items):\n    totals={key:sum(item[\'diagnostico\'][key] for item in items) for key in\n            [\'predicoes\',\'area_zero\',\'fora_da_imagem\',\'area_zero_na_grade\',\'fora_da_imagem_na_grade\']}\n    unusual=[item for item in items if item[\'diagnostico\'][\'area_zero\'] or item[\'diagnostico\'][\'fora_da_imagem\']]\n    report={\'correcao\':\'14-cache-validacao-1.0.1\',\'imagens\':len(items),\n        \'imagens_com_geometria_incomum\':len(unusual),\'totais\':totals,\n        \'propostas_removidas\':0,\'coordenadas_alteradas\':False,\n        \'observacao\':\'Geometria incomum preservada no cache e no protocolo; não constitui objeto aprovado. Reprodução em 0,25 continua obrigatória.\',\n        \'casos\':unusual}\n    core.save_json(Path(directory)/\'diagnostico_geometria_previsoes.json\',report)\n    return {k:v for k,v in report.items() if k!=\'casos\'}\n\ndef obtain_cache(model,records,directory,weight_sha,environment,device,predict_fn=None):\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    path=directory/\'predicoes_conf_0p001.jsonl.gz\';meta_path=directory/\'cache_manifesto.json\'\n    request=cache_request(records,weight_sha,environment)\n    if path.is_file() and meta_path.is_file():\n        meta=json.loads(meta_path.read_text())\n        if meta[\'solicitacao\']!=request or meta[\'cache_sha256\']!=core.sha256_file(path):\n            raise RuntimeError(\'Cache existente não corresponde aos pesos/entradas/ambiente. Não será reutilizado.\')\n        cached=read_cache(path,records)\n        diagnostics=[{\'id\':r[\'id\'],\'diagnostico\':validate_predictions(cached[r[\'id\']],r)} for r in records]\n        geometry=save_geometry_diagnostics(directory,diagnostics)\n        print(\'Cache conferido e reutilizado:\',records[0][\'coorte\'],flush=True)\n        return path,{**meta,\'geometria_resumo\':geometry,\'reutilizado_nesta_execucao\':True}\n    predict=predict_fn or core.model_predictions\n    tmp=path.with_name(path.name+\'.tmp\')\n    saturated=0;diagnostics=[]\n    with gzip.open(tmp,\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n        for i,r in enumerate(records,1):\n            if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Papel proibido na inferência.\')\n            pred=predict(model,r,device)\n            try:diagnostic=validate_predictions(pred,r)\n            except ValueError as exc:\n                core.save_json(directory/\'previsao_rejeitada.json\',{\'id\':r[\'id\'],\'mensagem\':str(exc),\n                    \'width\':r[\'width\'],\'height\':r[\'height\'],\'propostas\':pred})\n                raise\n            diagnostics.append({\'id\':r[\'id\'],\'diagnostico\':diagnostic})\n            if len(pred)==PREDICT_ARGS[\'max_det\']:saturated+=1\n            f.write(json.dumps({\'id\':r[\'id\'],\'imagem\':r[\'name\'],\'image_sha256\':r[\'sha256\'],\n                                \'label_sha256\':r[\'label_sha256\'],\'predicoes\':pred},ensure_ascii=False)+\'\\n\')\n            if i%100==0 or i==len(records):print(f"B_e03 / {r[\'coorte\']}: {i}/{len(records)}",flush=True)\n    tmp.replace(path)\n    geometry=save_geometry_diagnostics(directory,diagnostics)\n    meta={\'solicitacao\':request,\'cache_sha256\':core.sha256_file(path),\'concluido\':True,\n          \'validacao_previsoes\':\'14-cache-validacao-1.0.1\',\'geometria_resumo\':geometry,\n          \'imagens_atingiram_max_det\':saturated,\'limite\':\'Propostas pós-NMS em confiança 0,001; max_det=300, sem acesso aos logits brutos.\'}\n    core.save_json(meta_path,meta)\n    read_cache(path,records)\n    print(\'Diagnóstico geométrico do cache:\',geometry[\'totais\'],flush=True)\n    return path,{**meta,\'reutilizado_nesta_execucao\':False}\n', 'conf14_reports.py': '"""Relatórios automáticos e galeria do diagnóstico de confiança."""\nfrom __future__ import annotations\nimport html\nimport json\nfrom pathlib import Path\nimport zipfile\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.ticker import MaxNLocator\nfrom PIL import Image\nimport diag11_core as core\nimport diag11_reports as r11\nimport conf14_evaluation as evaluation\nimport conf14_data as data\n\nLABELS={\'home_val_historica\':\'Home VAL histórica\',\'openimages_val_nova\':\'Open Images VAL de desenvolvimento\'}\n\ndef pct(value):return \'—\' if value is None else f\'{100*value:.2f}%\'\n\ndef comparison_rows(results,bundle,selection):\n    rows=[]\n    for cohort in data.COHORTS:\n        anchor=results[cohort][.25][\'summary\'];a=anchor[\'alertas_por_objeto\']\n        for model in [\'baseline\',\'candidato_10\']:\n            s=bundle[\'evaluations\'][model][cohort][\'summary\'];m=s[\'alertas_por_objeto\']\n            rows.append({\'coorte\':cohort,\'modelo\':model,\'conf\':.25,\'origem\':\'registro autenticado do notebook 12\',\n                \'imagens\':s[\'imagens\'],**m,\'alta_ref\':s[\'alta_ref\'],\'alta_sem_alerta\':s[\'alta_sem_alerta\'],\n                \'delta_fp_vs_B_025\':m[\'fp\']-a[\'fp\'],\'delta_fn_vs_B_025\':m[\'fn\']-a[\'fn\'],\n                \'elegivel_14\':False})\n        for entry in selection[\'assessed\']:\n            conf=entry[\'conf\'];s=results[cohort][conf][\'summary\'];m=s[\'alertas_por_objeto\']\n            rows.append({\'coorte\':cohort,\'modelo\':\'B_e03\',\'conf\':conf,\'origem\':\'cache da nova inferência do notebook 14\',\n                \'imagens\':s[\'imagens\'],**m,\'alta_ref\':s[\'alta_ref\'],\'alta_sem_alerta\':s[\'alta_sem_alerta\'],\n                \'delta_fp_vs_B_025\':m[\'fp\']-a[\'fp\'],\'delta_fn_vs_B_025\':m[\'fn\']-a[\'fn\'],\n                \'elegivel_14\':entry[\'elegivel\']})\n    return rows\n\ndef plot_curves(output,results,bundle):\n    fig,axes=plt.subplots(4,2,figsize=(12,12),sharex=True,layout=\'constrained\')\n    for col,cohort in enumerate(data.COHORTS):\n        for row,key in enumerate([\'precision\',\'recall\',\'fp\',\'alta_sem_alerta\']):\n            values=[]\n            for conf in evaluation.THRESHOLDS:\n                s=results[cohort][conf][\'summary\']\n                v=s[key] if key==\'alta_sem_alerta\' else s[\'alertas_por_objeto\'][key]\n                values.append(100*v if row<2 and v is not None else v)\n            ax=axes[row,col];ax.plot(evaluation.THRESHOLDS,values,\'o-\',color=\'#176ca4\',label=\'B_e03\')\n            for model,color in [(\'baseline\',\'#6c7785\'),(\'candidato_10\',\'#d17920\')]:\n                s=bundle[\'evaluations\'][model][cohort][\'summary\']\n                v=s[key] if key==\'alta_sem_alerta\' else s[\'alertas_por_objeto\'][key]\n                if v is not None:ax.axhline(100*v if row<2 else v,color=color,linestyle=\'--\',label=model+\' / 0,25\')\n            ax.axvline(.25,color=\'#97a1ad\',linewidth=1,alpha=.6)\n            ax.grid(alpha=.2)\n            ax.set_ylabel([\'Precisão de alertas (%)\',\'Recall de alertas (%)\',\'Falsos alertas avaliáveis\',\'Objetos de prioridade alta sem alerta\'][row])\n            if row<2:ax.set_ylim(0,100)\n            else:\n                ax.set_ylim(bottom=0);ax.yaxis.set_major_locator(MaxNLocator(integer=True))\n        axes[0,col].set_title(LABELS[cohort]);axes[0,col].legend(fontsize=8)\n        axes[3,col].set_xticks(evaluation.THRESHOLDS)\n        axes[3,col].set_xlabel(\'Limiar global de confiança\')\n    fig.suptitle(\'Pesos B_e03 fixos · política V2 fixa · uma inferência por imagem\',fontsize=13)\n    fig.savefig(Path(output)/\'curvas_confianca.png\',dpi=150)\n    plt.close(fig)\n\ndef gallery(output,cohorts,caches,results,focus,policy,seed=42):\n    output=Path(output);index=[];cards=[]\n    for cohort,records in cohorts.items():\n        chosen=r11.select_gallery_cases(results[cohort][.25][\'images\'],results[cohort][focus][\'images\'],limit=12,seed=seed)\n        mapping={r[\'id\']:r for r in records}\n        for i,item in enumerate(chosen):\n            r=mapping[item[\'id\']];pred=caches[cohort][r[\'id\']]\n            gt=core.with_policy([g for g in r[\'gt\'] if r[\'known_mask\']&(1<<g[\'classe_id\'])],r[\'width\'],r[\'height\'],policy)\n            original=core.with_policy([p for p in pred if p[\'confianca\']>=.25],r[\'width\'],r[\'height\'],policy)\n            changed=core.with_policy([p for p in pred if p[\'confianca\']>=focus],r[\'width\'],r[\'height\'],policy)\n            panels=[r11.draw_panel(r,gt,\'Referência V2 avaliável\'),\n                    r11.draw_panel(r,original,\'B_e03 / confiança 0,25\'),\n                    r11.draw_panel(r,changed,f\'B_e03 / confiança {focus:.2f}\')]\n            canvas=Image.new(\'RGB\',(1620,470),\'#101a28\')\n            for j,panel in enumerate(panels):canvas.paste(panel,(j*540,0))\n            name=f\'galeria/{cohort}_{i:02}.jpg\';path=output/name;path.parent.mkdir(parents=True,exist_ok=True)\n            canvas.save(path,quality=86)\n            index.append({**item,\'coorte\':cohort,\'conf_comparada\':focus,\'arquivo\':name,\n                          \'creditos\':r.get(\'creditos\',{}),\'preenchimento_manual\':False})\n            cards.append(\'<article><h3>\'+html.escape(LABELS[cohort]+\' · \'+item[\'categoria\'])+\'</h3><img loading="lazy" src="\'+name+\'" alt="Referência e previsões do mesmo modelo em dois limiares"><p>\'+html.escape(r[\'name\'])+\'</p></article>\')\n    core.save_json(output/\'indice_galeria.json\',index)\n    return \'\'.join(cards)\n\ndef selection_table(selection):\n    head=[\'Confiança\',\'P Home\',\'R Home\',\'Alta omitida Home\',\'P externa\',\'R externa\',\'Alta omitida externa\',\'Cumpre todos os requisitos\']\n    table=\'<table><thead><tr>\'+\'\'.join(\'<th>\'+v+\'</th>\' for v in head)+\'</tr></thead><tbody>\'\n    for entry in selection[\'assessed\']:\n        h,e=entry[\'home\'],entry[\'external\'];ha,ea=h[\'alertas_por_objeto\'],e[\'alertas_por_objeto\']\n        values=[f"{entry[\'conf\']:.2f}",pct(ha[\'precision\']),pct(ha[\'recall\']),h[\'alta_sem_alerta\'],pct(ea[\'precision\']),pct(ea[\'recall\']),e[\'alta_sem_alerta\'],\'Sim, nas validações\' if entry[\'elegivel\'] else \'Não\']\n        table+=\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(v))+\'</td>\' for v in values)+\'</tr>\'\n    return table+\'</tbody></table>\'\n\ndef build_report(output,results,bundle,selection,reproduction,cohorts,caches,policy,config):\n    output=Path(output)\n    rows=comparison_rows(results,bundle,selection)\n    core.save_csv(output/\'comparacao_confianca.csv\',rows)\n    core.save_json(output/\'selecao_limiar.json\',selection)\n    checks=[{\'conf\':e[\'conf\'],**e[\'checks\'],\'comparavel_historico\':e[\'comparavel_historico\'],\n             \'elegivel_final\':e[\'elegivel\']} for e in selection[\'assessed\']]\n    core.save_csv(output/\'requisitos_por_limiar.csv\',checks)\n    changed=[];losses=[];priority=[]\n    for cohort in data.COHORTS:\n        a,b,c=evaluation.effects(results,bundle,cohort,caches[cohort]);changed+=a;losses+=b;priority+=c\n    core.save_csv(output/\'alertas_que_mudaram_vs_B_025.csv\',changed,\n        [\'coorte\',\'conf\',\'id\',\'classe\',\'bbox_ref\',\'alerta_B_025\',\'alerta_no_limiar\',\'prioridade_ref\',\'confianca_associada\'])\n    core.save_csv(output/\'perdas_vs_candidato10_por_limiar.csv\',losses,\n        [\'coorte\',\'conf\',\'id\',\'classe\',\'bbox_ref\',\'prioridade_ref\',\'recuperado\',\'evento_B_025\',\'evento_no_limiar\',\'melhor_iou_cache_mesma_classe\',\'confianca_dessa_proposta\',\'limite\'])\n    core.save_csv(output/\'perdas_com_objeto_ainda_detectado.csv\',priority,\n        [\'coorte\',\'id\',\'classe\',\'bbox_ref\',\'prioridade_candidato10\',\'prioridade_B_025\',\'confianca_candidato10\',\'confianca_B_025\',\'observacao\'])\n    recovery=[]\n    for cohort in data.COHORTS:\n        for conf in evaluation.THRESHOLDS:\n            part=[r for r in losses if r[\'coorte\']==cohort and r[\'conf\']==conf]\n            high=[r for r in part if r[\'prioridade_ref\']==\'alto\']\n            recovery.append({\'coorte\':cohort,\'conf\':conf,\'perdas_B_025_vs_candidato10\':len(part),\n                \'dessas_perdas_recuperadas\':sum(r[\'recuperado\'] for r in part),\n                \'perdas_altas_B_025_vs_candidato10\':len(high),\n                \'dessas_perdas_altas_recuperadas\':sum(r[\'recuperado\'] for r in high)})\n    core.save_csv(output/\'recuperacao_perdas_resumo.csv\',recovery)\n    plot_curves(output,results,bundle)\n    cards=gallery(output,cohorts,caches,results,selection[\'limiar_galeria\'],policy)\n    if not selection[\'comparabilidade_confirmada\']:\n        status=\'As contagens/associações em 0,25 divergiram do experimento 12. Os resultados são diagnósticos e nenhum limiar é sugerido.\'\n    elif selection[\'limiar_sugerido\'] is None:\n        status=\'Nenhum limiar da grade cumpriu todos os requisitos. O baseline continua em uso.\'\n    else:\n        status=f"O limiar {selection[\'limiar_sugerido\']:.2f} cumpriu os requisitos nas validações de desenvolvimento. É uma configuração candidata, sem substituição automática do baseline."\n    p=config[\'protocolo\'][\'regra_original_12\'];h=p[\'home\'];e=p[\'externa\']\n    refs=\'<table><tr><th>Validação</th><th>Referência em 0,25</th><th>Precisão</th><th>Recall</th><th>Alta sem alerta</th></tr>\'\n    for cohort in data.COHORTS:\n        for model in [\'baseline\',\'candidato_10\']:\n            s=bundle[\'evaluations\'][model][cohort][\'summary\'];a=s[\'alertas_por_objeto\']\n            refs+=\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(v))+\'</td>\' for v in [LABELS[cohort],model,pct(a[\'precision\']),pct(a[\'recall\']),s[\'alta_sem_alerta\']])+\'</tr>\'\n    refs+=\'</table>\'\n    reproduction_table=\'<table><tr><th>Validação</th><th>Divergências por imagem</th><th>Divergências por objeto</th><th>Reprodução confirmada</th></tr>\'\n    for r in reproduction:\n        reproduction_table+=\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(v))+\'</td>\' for v in [LABELS[r[\'coorte\']],r[\'divergencias_imagem\'],r[\'divergencias_objetos\'],\'Sim\' if r[\'reproduziu\'] else \'Não\'])+\'</tr>\'\n    reproduction_table+=\'</table>\'\n    css=\'body{font:16px/1.6 system-ui;background:#f3f6fa;color:#182638;margin:0}main{max-width:1320px;margin:auto;padding:28px}h1{line-height:1.2}h2{margin-top:30px}article,.note{background:white;border:1px solid #d6e0eb;border-radius:10px;padding:18px;margin:18px 0}table{border-collapse:collapse;background:white;width:100%;font-size:14px}th,td{border:1px solid #d6e0eb;padding:9px;text-align:left}.scroll{overflow:auto}img{width:100%;height:auto}a{color:#12689c}code{overflow-wrap:anywhere}\'\n    page=\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>TCC — Diagnóstico 14</title><style>\'+css+\'</style><main><h1>14 — Diagnóstico automático de confiança</h1>\'\n    page+=\'<div class="note"><strong>\'+html.escape(status)+\'</strong><p>Pesos B_e03 e política V2 fixos; zero novos treinos. Sem formulários. As referências do baseline e do candidato 10 são os registros autenticados do notebook 12, sem nova inferência desses dois modelos.</p></div>\'\n    page+=\'<h2>Grade fixa e requisitos preservados</h2><p>Um único limiar global deve funcionar nas duas validações. Home exige precisão superior a \'+pct(h[\'precision_maior_que\'])+\', recall de pelo menos \'+pct(h[\'recall_minimo\'])+\' e no máximo \'+str(h[\'alta_sem_alerta_maximo\'])+\' omissões altas. A externa exige precisão de pelo menos \'+pct(e[\'precision_minima\'])+\', recall de pelo menos \'+pct(e[\'recall_minimo\'])+\' e no máximo \'+str(e[\'alta_sem_alerta_maximo\'])+\' omissões altas. O JSON conserva os valores exatos.</p>\'\n    page+=\'<div class="scroll">\'+selection_table(selection)+\'</div><h2>Troca entre recuperação e falsos alertas</h2><img src="curvas_confianca.png" alt="Precisão, recall, falsos alertas e omissões altas em função do limiar">\'\n    page+=\'<p><a href="comparacao_confianca.csv">Comparação completa</a> · <a href="recuperacao_perdas_resumo.csv">Recuperação das perdas anteriores</a> · <a href="requisitos_por_limiar.csv">Requisitos por limiar</a></p>\'\n    page+=\'<p>As propostas numéricas do modelo foram preservadas, inclusive caixas de área zero. Os arquivos caches/&lt;coorte&gt;/diagnostico_geometria_previsoes.json registram geometrias incomuns. Não houve recorte ou remoção automática de propostas.</p>\'\n    page+=\'<h2>Reprodução em 0,25</h2><div class="scroll">\'+reproduction_table+\'</div><p>A confirmação exige igualdade dos contadores por imagem e dos estados por objeto. Qualquer divergência impede sugerir limiar por esses requisitos históricos; as diferenças ficam nos CSVs de reprodução.</p>\'\n    page+=\'<h2>Referências autenticadas do experimento 12</h2><div class="scroll">\'+refs+\'</div>\'\n    page+=\'<h2>Casos selecionados automaticamente</h2><p>Referência, B em 0,25 e B no limiar \'+f"{selection[\'limiar_galeria\']:.2f}"+\'. Sem limiar elegível, o painel usa 0,10, definido previamente apenas para diagnóstico. A galeria dirigida não estima prevalência de erros e não pede preenchimento.</p>\'+cards\n    page+=\'<h2>Limites</h2><p>Esta é seleção de ponto de operação em validações já usadas no desenvolvimento, não teste final independente nem prova de calibração probabilística. A origem histórica do treino do baseline permanece não determinada. Open Images tem cobertura parcial por imagem/classe; somente classes avaliáveis entram nas métricas. Confiança menor pode aumentar falsos alertas e alterar associações. As previsões são pós-NMS com max_det=300. A política V2 nas caixas não constitui validação independente de risco físico. As 300 imagens reservadas e os testes ficaram sem extração/inferência nesta etapa. Os rótulos não foram modificados; a auditoria anterior por IA não é uma aprovação humana da base.</p></main></html>\'\n    (output/\'relatorio_diagnostico_14.html\').write_text(page,encoding=\'utf-8\')\n    summary={\'versao\':evaluation.VERSION,\'concluido\':True,\'status\':selection[\'status\'],\'mensagem\':status,\n        \'limiar_sugerido\':selection[\'limiar_sugerido\'],\'grade\':list(evaluation.THRESHOLDS),\n        \'reproducao_confirmada\':selection[\'comparabilidade_confirmada\'],\n        \'modelo_inferido\':\'B_e03\',\'imagens_avaliadas\':sum(len(v) for v in cohorts.values()),\n        \'imagens_inferidas_nesta_execucao\':config.get(\'imagens_inferidas_nesta_execucao\'),\n        \'novos_treinamentos\':0,\'preenchimento_manual_obrigatorio\':False,\n        \'baseline_e_candidato10_reinferidos\':False,\'referencias\':\'pacote 12 autenticado\',\n        \'modelo_em_uso_continua_baseline\':True,\'modelos_substituidos\':False,\n        \'reserva_futura_extraida\':False,\'reserva_futura_inferida\':False,\n        \'test_home_inferido\':False,\'test_oi20k_inferido\':False,\'rotulos_alterados\':False,\n        \'validacoes_usadas_na_selecao\':True,\'nao_e_teste_final_independente\':True,\n        \'correcao_cache\':\'14-cache-validacao-1.0.1\',\n        \'geometria_previsoes\':{c:m.get(\'geometria_resumo\',{}) for c,m in config.get(\'caches\',{}).items()},\n        \'peso_B_e03_sha256\':config[\'peso_B_e03_sha256\'],\'dataset_sha256\':config[\'dataset_sha256\'],\n        \'recuperacao_perdas\':recovery,\n        \'proximo_passo\':(\'Congelar a configuração candidata e preparar avaliação futura independente, sem uso automático da reserva.\' if selection[\'limiar_sugerido\'] is not None else\n                        \'Analisar as diferenças de reprodução antes de selecionar configuração.\' if not selection[\'comparabilidade_confirmada\'] else\n                        \'Encerrar esta hipótese de limiar; planejar revisão automática mais ampla dos dados e novo experimento documentado.\')}\n    core.save_json(output/\'resumo_diagnostico_14.json\',summary)\n    (output/\'LEIA_RESULTADO.md\').write_text(\'# Diagnóstico 14\\n\\n\'+status+\'\\n\\nAbra relatorio_diagnostico_14.html. As comparações e a galeria já foram geradas; não há fichas para preencher. Resultados em pacote_diagnostico_14.zip para documentação.\\n\\nO pacote inclui as previsões completas pós-NMS em confiança 0,001, para permitir novas análises sem repetir a inferência. Nenhum treinamento foi executado ou modelo em uso foi substituído.\\n\',encoding=\'utf-8\')\n    return summary\n\ndef package_reports(output):\n    output=Path(output);archive=output/\'pacote_diagnostico_14.zip\'\n    with zipfile.ZipFile(archive,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file() and path!=archive and not path.name.endswith(\'.tmp\') and path.suffix not in {\'.pt\',\'.pkl\'}:\n                z.write(path,path.relative_to(output))\n    return archive\n'}
ETAPAS_RETORNO = {'inferir': "CACHE_FILES={};CACHE_METADADOS={};CACHES={}\ntry:\n    MODELO=YOLO(str(PESO_B))\n    if {int(k):v for k,v in MODELO.names.items()}!=dict(enumerate(core.NAMES)):\n        raise RuntimeError('As classes dos pesos não correspondem às 12 classes do projeto.')\n    for coorte,registros in COORTES.items():\n        cache,meta=avaliacao14.obtain_cache(MODELO,registros,PASTA_RESULTADOS/'caches'/coorte,\n            EXPECTED['B_e03'],AMBIENTE,DEVICE)\n        CACHE_FILES[coorte]=cache;CACHE_METADADOS[coorte]=meta\n        CACHES[coorte]=avaliacao14.read_cache(cache,registros)\n        FROZEN_FILES += [(cache,core.sha256_file(cache)),(cache.parent/'cache_manifesto.json',core.sha256_file(cache.parent/'cache_manifesto.json'))]\n        core.save_json(PASTA_RESULTADOS/'caches_concluidos.json',CACHE_METADADOS)\n        enviar_resultados('cache concluído: '+coorte)\n    del MODELO\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    CONFIG_14['caches']=CACHE_METADADOS\n    CONFIG_14['imagens_inferidas_nesta_execucao']=sum(\n        meta['solicitacao']['imagens'] for meta in CACHE_METADADOS.values() if not meta['reutilizado_nesta_execucao'])\n    core.save_json(PASTA_RESULTADOS/'configuracao_diagnostico_14.json',CONFIG_14)\n    print('Inferência concluída. A varredura seguinte usa os caches salvos.')\nexcept Exception as exc:\n    registrar_falha(exc,'inferência')\n    raise\n", 'avaliar': "RESULTADOS={};REPRODUCAO=[];REPRO_IMAGENS=[];REPRO_OBJETOS=[]\ntry:\n    for coorte,registros in COORTES.items():\n        RESULTADOS[coorte]={}\n        base=PASTA_RESULTADOS/'avaliacoes'/coorte\n        resultado=avaliacao14.evaluate_cache(registros,CACHES[coorte],.25,politica,\n            avaliacao_base.corresponder_objetos,base/avaliacao14.point_name(.25))\n        RESULTADOS[coorte][.25]=resultado\n        resumo,imagens,objetos=avaliacao14.check_reproduction(\n            resultado,BUNDLE['evaluations']['B_e03'][coorte],coorte)\n        REPRODUCAO.append(resumo);REPRO_IMAGENS+=imagens;REPRO_OBJETOS+=objetos\n        print(coorte,'| reprodução em 0,25:',resumo['reproduziu'],flush=True)\n        for conf in avaliacao14.THRESHOLDS:\n            if conf==.25:continue\n            RESULTADOS[coorte][conf]=avaliacao14.evaluate_cache(registros,CACHES[coorte],conf,\n                politica,avaliacao_base.corresponder_objetos,base/avaliacao14.point_name(conf))\n            a=RESULTADOS[coorte][conf]['summary']['alertas_por_objeto']\n            print(coorte,'| confiança',conf,'| TP',a['tp'],'FN',a['fn'],'FP',a['fp'],flush=True)\n        core.save_json(PASTA_RESULTADOS/'reproducao_025_resumo.json',REPRODUCAO)\n        core.save_csv(PASTA_RESULTADOS/'reproducao_025_por_imagem.csv',REPRO_IMAGENS)\n        core.save_csv(PASTA_RESULTADOS/'reproducao_025_por_objeto.csv',REPRO_OBJETOS)\n        enviar_resultados('grade avaliada: '+coorte)\n    REPRODUZIU=all(r['reproduziu'] for r in REPRODUCAO)\n    SELECAO=avaliacao14.select_threshold(RESULTADOS,BUNDLE['rule'],REPRODUZIU)\n    print('Resultado da seleção:',SELECAO['status'])\n    print('Limiar candidato nas validações:',SELECAO['limiar_sugerido'])\nexcept Exception as exc:\n    registrar_falha(exc,'avaliação da grade')\n    raise\n", 'relatorio': "try:\n    dados12.verify_frozen(FROZEN_FILES)\n    RESUMO=relatorios14.build_report(PASTA_RESULTADOS,RESULTADOS,BUNDLE,SELECAO,\n        REPRODUCAO,COORTES,CACHES,politica,CONFIG_14)\n    core.save_json(PASTA_RESULTADOS/'integridade_final.json',{\n        'conferida':True,'arquivos_conferidos':len(FROZEN_FILES),\n        'rotulos_alterados':False,'pesos_originais_preservados':True,\n        'reserva_futura_extraida':False,'reserva_futura_inferida':False,\n        'novos_treinamentos':0,'preenchimento_manual_obrigatorio':False})\n    # Uma falha anterior resolvida fica no histórico, fora do estado atual.\n    erro_anterior=PASTA_RESULTADOS/'execucao_interrompida.json'\n    if erro_anterior.is_file():\n        historico=PASTA_RESULTADOS/'historico_falhas';historico.mkdir(exist_ok=True)\n        erro_anterior.replace(historico/('execucao_interrompida_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.json'))\n    PACOTE_FINAL=relatorios14.package_reports(PASTA_RESULTADOS)\n    enviar_resultados('pacote final do diagnóstico 14')\n    print(RESUMO['mensagem'])\n    print('Diagnóstico finalizado. Arquivo de resultados:',PACOTE_FINAL)\n    display(FileLink(str(PACOTE_FINAL)))\n    display(FileLink(str(PASTA_RESULTADOS/'relatorio_diagnostico_14.html')))\n    display(FileLink(str(PASTA_RESULTADOS/'comparacao_confianca.csv')))\nexcept Exception as exc:\n    registrar_falha(exc,'relatório e salvamento')\n    raise\n"}

for nome,fonte in CODIGO_CORRIGIDO.items():
    modulo=avaliacao14 if nome=='conf14_evaluation.py' else relatorios14
    arquivo=Path(PASTA_CODIGO)/nome
    compile(fonte,str(arquivo),'exec')
    arquivo.write_text(fonte,encoding='utf-8')
    exec(compile(fonte,str(arquivo),'exec'),modulo.__dict__)
    MODULOS_EMBUTIDOS[nome]=fonte
    destino=Path(PASTA_RESULTADOS)/'codigo'/nome
    destino.parent.mkdir(parents=True,exist_ok=True)
    destino.write_text(fonte,encoding='utf-8')
    CONFIG_14['codigo_embutido_sha256'][nome]=hashlib.sha256(fonte.encode()).hexdigest()
CONFIG_14['correcao_cache']={'id': '14-cache-validacao-1.0.1', 'origem': 'Correção da checagem da etapa 6', 'conserva_propostas_numericas': True, 'altera_pesos': False, 'altera_rotulos': False, 'altera_grade': False, 'altera_associador': False, 'reproducao_025_mantida': True}
core.save_json(Path(PASTA_RESULTADOS)/'configuracao_diagnostico_14.json',CONFIG_14)
core.save_json(Path(PASTA_RESULTADOS)/'correcao_etapa6_aplicada.json',{
    **CONFIG_14['correcao_cache'],'aplicada_em':datetime.now(timezone.utc).isoformat(),
    'sessao_reutilizada':True,'dados_baixados_novamente':False})
if 'MODELO' in globals():
    del MODELO
    if torch.cuda.is_available():torch.cuda.empty_cache()
print('Correção aplicada. Retomando da etapa 6, com os dados já baixados.',flush=True)
for etapa in ['inferir','avaliar','relatorio']:
    exec(compile(ETAPAS_RETORNO[etapa],'retomada14-'+etapa,'exec'),globals())
